# Clase 16 — SOLUCIÓN de Ejercicios

**Diplomado en Data Science Aplicada con Python para la Toma de Decisiones**  
Arca Continental Ecuador | UDLA  
*15 de Abril, 2026*

Este notebook contiene las soluciones de los 6 ejercicios de la clase 16.

## Setup

In [ ]:
# !pip install plotly
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             confusion_matrix, roc_curve, roc_auc_score,
                             precision_recall_curve, average_precision_score)

sns.set_theme(style="whitegrid", context="notebook")

URL = "https://raw.githubusercontent.com/cmosquerat/arca-diplomado/main/clase-16/stroke.csv"
df = pd.read_csv(URL)
df = df.drop(columns=["id"])
df["bmi"] = df["bmi"].fillna(df["bmi"].median())
df_enc = pd.get_dummies(df, drop_first=True, dtype=int)
X = df_enc.drop(columns=["stroke"])
y = df_enc["stroke"]
print(f"Dataset: {df.shape}, positivos: {y.mean()*100:.2f}%")

---

## Ejercicio 1: Primer contacto con Plotly

In [ ]:
df_gap = px.data.gapminder()

# 1. Violin plot: esperanza de vida por continente
fig = px.violin(df_gap, x="continent", y="lifeExp",
                color="continent",
                box=True, points="all",
                title="Esperanza de vida por continente (1952-2007)")
fig.update_layout(template="plotly_dark", width=850, height=550)
fig.show()

In [ ]:
# 3. Animacion temporal de Rosling - el superpoder de Plotly
fig = px.scatter(df_gap, x="gdpPercap", y="lifeExp",
                 animation_frame="year", animation_group="country",
                 color="continent", size="pop", hover_name="country",
                 log_x=True, size_max=60, range_y=[25, 90],
                 title="El progreso del mundo (Hans Rosling) — dale play")
fig.show()

# 4. Choropleth: mapa mundial
gap_2007 = df_gap[df_gap["year"] == 2007]
fig = px.choropleth(gap_2007, locations="iso_alpha",
                    color="lifeExp", hover_name="country",
                    color_continuous_scale="RdYlGn",
                    title="Esperanza de vida en 2007 (mapa mundial)")
fig.show()

---

## Ejercicio 1b: Scatter 3D rotable

In [ ]:
# 1-2. Scatter 3D coloreado por stroke
fig = px.scatter_3d(df, x="age", y="avg_glucose_level", z="bmi",
                    color="stroke", opacity=0.6,
                    color_continuous_scale=["#2563EB", "#C82B40"],
                    title="Stroke en 3D — rota con el mouse")
fig.show()

**Patrón visible:** los puntos rojos (stroke=1) se concentran en la zona de **edad alta** y **glucosa alta**. El BMI muestra menos separación.

In [ ]:
# 3. Coloreado por hipertension
fig = px.scatter_3d(df, x="age", y="avg_glucose_level", z="bmi",
                    color="hypertension", opacity=0.6,
                    title="Hipertension en 3D")
fig.show()

In [ ]:
# 4. Bonus: tamanio por edad
fig = px.scatter_3d(df, x="age", y="avg_glucose_level", z="bmi",
                    color="stroke", size="age", opacity=0.5,
                    size_max=15,
                    title="Stroke en 3D con tamano=edad")
fig.show()

---

## Ejercicio 1c: Explora otros tipos

In [ ]:
# 1. Sunburst
df_s = df.copy()
df_s["stroke_lbl"] = df_s["stroke"].map({0: "Sano", 1: "ACV"})
fig = px.sunburst(df_s, path=["smoking_status", "gender", "stroke_lbl"],
                  title="Smoking -> Gender -> Stroke")
fig.show()

In [ ]:
# 3. Parallel coordinates
fig = px.parallel_coordinates(
    df[["age", "avg_glucose_level", "bmi", "stroke"]],
    color="stroke",
    color_continuous_scale=["#2563EB", "#C82B40"],
    title="Parallel coordinates — las lineas rojas son los casos con ACV")
fig.show()

In [ ]:
# 4. Density heatmap
fig = px.density_heatmap(df, x="age", y="avg_glucose_level",
                         nbinsx=25, nbinsy=25,
                         title="Densidad de pacientes: edad vs glucosa",
                         color_continuous_scale="Reds")
fig.show()

In [ ]:
# 5. Violin de BMI por stroke
fig = px.violin(df, x="stroke", y="bmi", color="stroke",
                box=True, points="outliers",
                color_discrete_sequence=["#2563EB", "#C82B40"],
                title="BMI por estado de ACV")
fig.show()

---

## Ejercicio 2: Tu primer split honesto

In [ ]:
# 1-2. Split con test_size=0.3 y verificacion
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=7, stratify=y)

print(f"Train: {len(X_train)} ({y_train.mean()*100:.2f}% positivos)")
print(f"Test:  {len(X_test)} ({y_test.mean()*100:.2f}% positivos)")

In [ ]:
# 3. Escalar solo con train
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# 4-5. Dos modelos
m_plain = LogisticRegression(max_iter=1000)
m_plain.fit(X_train_s, y_train)
pred_plain = m_plain.predict(X_test_s)

m_bal = LogisticRegression(max_iter=1000, class_weight="balanced")
m_bal.fit(X_train_s, y_train)
pred_bal = m_bal.predict(X_test_s)

print(f"{'Modelo':<12} {'Precision':>10} {'Recall':>8}")
print("-" * 35)
print(f"{'Plain':<12} {precision_score(y_test, pred_plain, zero_division=0):>10.3f} {recall_score(y_test, pred_plain):>8.3f}")
print(f"{'Balanced':<12} {precision_score(y_test, pred_bal, zero_division=0):>10.3f} {recall_score(y_test, pred_bal):>8.3f}")

**Respuesta 6:** el modelo **balanced** tiene recall mucho más alto (~0.75 vs ~0.0) pero a costa de la precisión (~0.14 vs indefinido / 0).

**Interpretación clínica:** el modelo plain prácticamente nunca dice "ACV" porque optimiza accuracy y el desbalance es 95/5. El balanced es el que realmente sirve: detecta ~75% de los ACV reales, aunque a 1 de cada 7 falsos positivos tocará hacerle un chequeo innecesario. Para medicina, es un trade-off aceptable.

---

## Ejercicio 3: Tu curva ROC y PR

In [ ]:
# Usamos el split del 80/20 de la clase principal
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# 1. Modelo sin balancear
modelo = LogisticRegression(max_iter=1000)
modelo.fit(X_train_s, y_train)

# 2. Probabilidades
y_proba = modelo.predict_proba(X_test_s)[:, 1]
print(f"Primeras 5 probabilidades: {np.round(y_proba[:5], 3)}")

In [ ]:
# 3. Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)

fig = px.area(x=fpr, y=tpr,
              title=f"Curva ROC (AUC = {auc:.3f})",
              labels={"x": "FPR", "y": "TPR (Recall)"},
              width=700, height=500)
fig.add_shape(type="line", line=dict(dash="dash", color="gray"),
              x0=0, x1=1, y0=0, y1=1)
fig.update_layout(template="plotly_white")
fig.show()
print(f"AUC = {auc:.3f}")

In [ ]:
# 4. Curva PR
prec, rec, _ = precision_recall_curve(y_test, y_proba)
ap = average_precision_score(y_test, y_proba)

fig = px.area(x=rec, y=prec,
              title=f"Curva Precision-Recall (AP = {ap:.3f})",
              labels={"x": "Recall", "y": "Precision"},
              width=700, height=500)
fig.add_hline(y=y_test.mean(), line_dash="dash", line_color="gray",
              annotation_text=f"baseline = {y_test.mean():.3f}")
fig.update_layout(template="plotly_white")
fig.show()
print(f"AP = {ap:.3f}")

In [ ]:
# 5. Cinco umbrales
print(f"{'Umbral':>8} {'Prec':>8} {'Recall':>8} {'FN':>5} {'FP':>5} {'VP':>5}")
print("-" * 45)
for u in [0.1, 0.3, 0.5, 0.7, 0.9]:
    y_pred_u = (y_proba >= u).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_u).ravel()
    p = precision_score(y_test, y_pred_u, zero_division=0)
    r = recall_score(y_test, y_pred_u)
    print(f"{u:>8.2f} {p:>8.3f} {r:>8.3f} {fn:>5d} {fp:>5d} {tp:>5d}")

In [ ]:
# 6. Umbral optimo con FN=50x FP
COSTO_FN = 50
COSTO_FP = 1

mejores = []
for u in np.arange(0.02, 0.99, 0.01):
    y_pred_u = (y_proba >= u).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_u).ravel()
    mejores.append((u, fn*COSTO_FN + fp*COSTO_FP))
df_cost = pd.DataFrame(mejores, columns=["umbral", "costo"])
u_opt = df_cost.loc[df_cost["costo"].idxmin(), "umbral"]
print(f"Umbral optimo con costo FN=50x: {u_opt:.2f}")

**Respuesta 6:** el umbral óptimo cuando el FN cuesta 50 veces el FP está típicamente entre 0.05 y 0.15 — **muy por debajo del 0.5 por defecto**. En términos clínicos: ante la duda, mandamos al paciente al chequeo.

---

## Ejercicio 4 (retador): Comparar dos modelos

In [ ]:
# 1-2. Entrenar dos modelos y obtener probabilidades
modelo_a = LogisticRegression(max_iter=1000)
modelo_a.fit(X_train_s, y_train)
proba_a = modelo_a.predict_proba(X_test_s)[:, 1]

modelo_b = LogisticRegression(max_iter=1000, class_weight="balanced")
modelo_b.fit(X_train_s, y_train)
proba_b = modelo_b.predict_proba(X_test_s)[:, 1]

# 3. ROC de ambos en el mismo grafico
fpr_a, tpr_a, _ = roc_curve(y_test, proba_a)
fpr_b, tpr_b, _ = roc_curve(y_test, proba_b)
auc_a = roc_auc_score(y_test, proba_a)
auc_b = roc_auc_score(y_test, proba_b)

fig = go.Figure()
fig.add_trace(go.Scatter(x=fpr_a, y=tpr_a, mode="lines",
                         name=f"Plain (AUC={auc_a:.3f})",
                         line=dict(color="#2563EB", width=3)))
fig.add_trace(go.Scatter(x=fpr_b, y=tpr_b, mode="lines",
                         name=f"Balanced (AUC={auc_b:.3f})",
                         line=dict(color="#C82B40", width=3)))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines",
                         name="Azar", line=dict(dash="dash", color="gray")))
fig.update_layout(title="Curva ROC — Plain vs Balanced",
                  xaxis_title="FPR", yaxis_title="TPR",
                  template="plotly_white", width=750, height=550)
fig.show()

In [ ]:
# 4. PR de ambos en el mismo grafico
prec_a, rec_a, _ = precision_recall_curve(y_test, proba_a)
prec_b, rec_b, _ = precision_recall_curve(y_test, proba_b)
ap_a = average_precision_score(y_test, proba_a)
ap_b = average_precision_score(y_test, proba_b)

fig = go.Figure()
fig.add_trace(go.Scatter(x=rec_a, y=prec_a, mode="lines",
                         name=f"Plain (AP={ap_a:.3f})",
                         line=dict(color="#2563EB", width=3)))
fig.add_trace(go.Scatter(x=rec_b, y=prec_b, mode="lines",
                         name=f"Balanced (AP={ap_b:.3f})",
                         line=dict(color="#C82B40", width=3)))
fig.add_hline(y=y_test.mean(), line_dash="dash", line_color="gray",
              annotation_text="baseline")
fig.update_layout(title="Curva PR — Plain vs Balanced",
                  xaxis_title="Recall", yaxis_title="Precision",
                  template="plotly_white", width=750, height=550)
fig.show()

print(f"AUC plain={auc_a:.3f}   AUC balanced={auc_b:.3f}")
print(f"AP  plain={ap_a:.3f}   AP  balanced={ap_b:.3f}")

### Respuesta 5 — Análisis

**Observación clave:** el AUC de ambos modelos es **casi idéntico**. El AP también lo es.

**¿Por qué?** Tanto ROC como PR se calculan sobre **todas las probabilidades ordenadas**, no sobre las predicciones binarias. `class_weight="balanced"` cambia cómo el modelo pondera los errores al entrenar, pero el **ordenamiento** de las probabilidades no cambia mucho — solo se desplazan.

**Conclusión:** las curvas ROC y PR son **invariantes al desbalance del umbral**. Lo que cambia con `class_weight="balanced"` es dónde cae la **predicción binaria** (el punto específico sobre la curva), no la curva completa.

**Lección:** para comparar modelos usen AUC/AP. Para elegir el umbral operacional, usen la curva + costo de negocio.

---

## Fin de soluciones